# Масштабируемый ML в PySpark

Классический стек машинного обучения работает в рамках одного процесса и единой оперативной памяти. Как только объем данных превышает размер доступной RAM на вычислительном узле, вы неизбежно сталкиваетесь с ошибкой `Out of Memory` (OOM).

`PySpark ML` — это библиотека для распределенного машинного обучения. Она позволяет разбивать данные на партиции и обрабатывать их параллельно, задействуя все доступные ядра процессора или ресурсы кластера.

### Два главных правила работы со Spark:
1. **Lazy Evaluation (Ленивые вычисления):** Spark не выполняет трансформации данных мгновенно. Вместо этого он строит логический план вычислений — направленный ациклический граф задач (DAG). Реальный расчет запускается только при вызове «действия» (Action), такого как `.show()`, `.count()` или `.fit()`.
2. **Запрет на `.toPandas()` на больших объемах:** Этот метод принудительно выгружает все распределенные данные с кластера в память вашей текущей Jupyter-сессии. На реальных данных это приведет к падению сервера. Для анализа структуры и содержимого всегда используйте встроенный метод `df.show()`.

In [1]:
import os
from pyspark.sql import SparkSession

# Инициализируем локальную сессию Spark, используя все доступные ядра процессора [*]
spark = SparkSession.builder \
    .appName("PySpark_ML_Demo") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Session успешно создана! Версия Spark: {spark.version}")

Spark Session успешно создана! Версия Spark: 3.5.0


Параметр master определяет, где именно будут выполняться вычисления. Слово `local` означает, что мы запускаем Spark локально на одной машине (внутри нашего контейнера), а звездочка `[*]` указывает Спарку автоматически определить количество ядер (threads) у процессора компьютера и задействовать их все для параллельных вычислений. Если бы мы написали `local[2]`, то Spark использовал бы строго 2 ядра.

### Исследуемый датасет: Отток клиентов банка (Bank Churn)
Мы анализируем исторические данные о поведении клиентов банка. Наша задача — спрогнозировать вероятность ухода клиента. Целевая переменная записана в столбце `Exited` (1 — клиент ушел, 0 — остался).

Основные признаки:
* `CreditScore` — кредитный рейтинг
* `Geography` — страна проживания (категориальный)
* `Gender` — пол (категориальный)
* `Age` — возраст
* `Tenure` — сколько лет клиент обслуживается в банке
* `Balance` — баланс на счете
* `NumOfProducts` — количество используемых продуктов банка
* `EstimatedSalary` — предполагаемая заработная плата

In [2]:
# Читаем данные с автоматическим определением типов колонок
(df := spark.read.csv("data/Churn_Modelling.csv", header=True, inferSchema=True))

DataFrame[RowNumber: int, CustomerId: int, Surname: string, CreditScore: int, Geography: string, Gender: string, Age: int, Tenure: int, Balance: double, NumOfProducts: int, HasCrCard: int, IsActiveMember: int, EstimatedSalary: double, Exited: int]

In [3]:
# Смотрим структуру данных
df.printSchema()

root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)



In [4]:
# Выводим первые 5 строк для визуального ознакомления
df.show(5)

+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId| Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        1|  15634602|Hargrave|        619|   France|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|
|        2|  15647311|    Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        3|  15619304|    Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        4|  15701354|    Boni|        699|   France|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|

In [5]:
# Расчет базовых статистик для числовых признаков
df.select("CreditScore", "Age", "Balance", "EstimatedSalary").describe().show()

+-------+-----------------+------------------+-----------------+-----------------+
|summary|      CreditScore|               Age|          Balance|  EstimatedSalary|
+-------+-----------------+------------------+-----------------+-----------------+
|  count|            10000|             10000|            10000|            10000|
|   mean|         650.5288|           38.9218|76485.88928799961|100090.2398809998|
| stddev|96.65329873613035|10.487806451704587|62397.40520238599|57510.49281769821|
|    min|              350|                18|              0.0|            11.58|
|    max|              850|                92|        250898.09|        199992.48|
+-------+-----------------+------------------+-----------------+-----------------+



### Кодирование категориальных признаков

Текстовые колонки `Geography` и `Gender` необходимо преобразовать. Для этого в PySpark используется инструмент `StringIndexer`, который сопоставляет каждую текстовую категорию с числовым индексом.

**Возможная инженерная проблема продакшена (Data Drift):**
Представьте ситуацию: на этапе обучения модель видела в колонке `Geography` только страны `France`, `Spain` и `Germany`. Если на этапе тестирования или в реальном продакшене в систему прилетит объект с новой, ранее неизвестной категорией (например, `Italy`), стандартный `StringIndexer` упадет с критической ошибкой, остановив работу всего конвейера.

Чтобы защитить пайплайн от таких сбоев, используется параметр `handleInvalid="keep"`. В этом случае Spark не падает, а автоматически присваивает всем новым и незнакомым категориям резервный индекс.

In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Инициализируем индексаторы с защитой от неизвестных категорий
geo_indexer = StringIndexer(inputCol="Geography", outputCol="GeographyIndex", handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex", handleInvalid="keep")

# Последовательно обучаем индексаторы и трансформируем датасет
df_indexed = geo_indexer.fit(df).transform(df)
df_indexed = gender_indexer.fit(df_indexed).transform(df_indexed)

# Применяем OneHotEncoder для перевода индексов в бинарные векторы
encoder = OneHotEncoder(
    inputCols=["GeographyIndex", "GenderIndex"],
    outputCols=["GeographyVec", "GenderVec"]
)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# Изучаем результат трансформации
df_encoded.select("Geography", "GeographyIndex", "GeographyVec", "Gender", "GenderVec").show(5, truncate=False)

+---------+--------------+-------------+------+-------------+
|Geography|GeographyIndex|GeographyVec |Gender|GenderVec    |
+---------+--------------+-------------+------+-------------+
|France   |0.0           |(3,[0],[1.0])|Female|(2,[1],[1.0])|
|Spain    |2.0           |(3,[2],[1.0])|Female|(2,[1],[1.0])|
|France   |0.0           |(3,[0],[1.0])|Female|(2,[1],[1.0])|
|France   |0.0           |(3,[0],[1.0])|Female|(2,[1],[1.0])|
|Spain    |2.0           |(3,[2],[1.0])|Female|(2,[1],[1.0])|
+---------+--------------+-------------+------+-------------+
only showing top 5 rows


### Объединение признаков (VectorAssembler)

В большинстве классических библиотек ML алгоритмам на вход передается двумерная матрица или таблица признаков (X). В `PySpark ML` все модели ожидают на вход одинстолбец, внутри которого все признаки упакованы в специальный тип данных — Vector.

Для объединения всех числовых колонок и полученных после OneHotEncoder векторов в одну структуру используется трансформер `VectorAssembler`.

In [7]:
# Формируем итоговый список колонок, которые станут признаками для обучения
feature_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "GeographyVec", "GenderVec"
]

# Склеиваем колонки в единый системный столбец "features"
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
final_data = assembler.transform(df_encoded)

final_data.select("features", "Exited").show(5, truncate=False)

+-------------------------------------------------------------------+------+
|features                                                           |Exited|
+-------------------------------------------------------------------+------+
|[619.0,42.0,2.0,0.0,1.0,1.0,1.0,101348.88,1.0,0.0,0.0,0.0,1.0]     |1     |
|[608.0,41.0,1.0,83807.86,1.0,0.0,1.0,112542.58,0.0,0.0,1.0,0.0,1.0]|0     |
|[502.0,42.0,8.0,159660.8,3.0,1.0,0.0,113931.57,1.0,0.0,0.0,0.0,1.0]|1     |
|(13,[0,1,2,4,7,8,12],[699.0,39.0,1.0,2.0,93826.63,1.0,1.0])        |0     |
|[850.0,43.0,2.0,125510.82,1.0,1.0,1.0,79084.1,0.0,0.0,1.0,0.0,1.0] |0     |
+-------------------------------------------------------------------+------+
only showing top 5 rows


Чтобы не тратить оперативную память на хранение повторяющихся нулей, Spark оптимизирует структуру и использует разреженные векторы (Sparse Vectors).
Запись вида `(11, [0,1], [619.0, 42.0])` означает: перед нами вектор из 11 элементов, в котором только на позициях 0 и 1 стоят реальные значения (619.0 и 42.0), а все остальные позиции заполнены нулями.

### Сборка единого конвейера (Pipeline)

Мы разобрали пошаговую подготовку данных вручную. Однако в production-системах такой подход считается неэффективным: он загромождает код промежуточным датафреймами и повышает риск ошибки. Профессиональный стандарт в Big Data — объединение всех шагов трансформации и самой модели классификатора в сквозной **Pipeline**.

Мы разделяем сырые данные на обучающую и тестовую выборки в самом начале (инженерный стандарт контроля утечки данных), а затем обучаем `.fit()` и применяем `.transform()` весь конвейер целиком.

In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Делим исходный датасет на train/test до начала трансформаций
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 2. Объявляем компоненты нашего будущего конвейера
geo_indexer = StringIndexer(inputCol="Geography", outputCol="GeographyIndex", handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex", handleInvalid="keep")

encoder = OneHotEncoder(inputCols=["GeographyIndex", "GenderIndex"], outputCols=["GeographyVec", "GenderVec"])

feature_cols = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
    "HasCrCard", "IsActiveMember", "EstimatedSalary", "GeographyVec", "GenderVec"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Инициализируем алгоритм машинного обучения
rf = RandomForestClassifier(featuresCol="features", labelCol="Exited", numTrees=20, seed=42)

# 3. Собираем компоненты в единый Pipeline в правильной последовательности
pipeline = Pipeline(stages=[geo_indexer, gender_indexer, encoder, assembler, rf])

# 4. Обучаем весь конвейер одной командой на обучающей выборке
pipeline_model = pipeline.fit(train_df)

# 5. Пропускаем тестовую выборку через обученный конвейер
predictions = pipeline_model.transform(test_df)

# 6. Оцениваем итоговое качество работы модели
evaluator = MulticlassClassificationEvaluator(labelCol="Exited", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Итоговая точность (Accuracy) пайплайна: {accuracy * 100:.2f}%")

Итоговая точность (Accuracy) пайплайна: 85.06%


In [10]:
# Сохраняем пайплайн
pipeline_model.write().overwrite().save("demo_pipeline")

In [12]:
# Загружаем сохранённый пайплайн
from pyspark.ml import PipelineModel

loaded = PipelineModel.load("demo_pipeline")

loaded.transform(test_df).show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+--------------+-----------+-------------+-------------+--------------------+--------------------+--------------------+----------+
|RowNumber|CustomerId|  Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|GeographyIndex|GenderIndex| GeographyVec|    GenderVec|            features|       rawPrediction|         probability|prediction|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+--------------+-----------+-------------+-------------+--------------------+--------------------+--------------------+----------+
|        3|  15619304|     Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|           0.0|        1.0|(3,[0],